# SPATIAL INTELLIGENCE - CENTRALITY ANALYSIS

Analyze the prison floor plan using closeness centrality (integration) and betweenness centrality (choice) from Space Syntax theory.

## 1. Import the needed libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy version

In [ ]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Configuration

In [ ]:
import os

renderer = "vscode"

# Paths
BASE_DIR = r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Assignment-02_RamonGarcia"
BREP_PATH = os.path.join(BASE_DIR, "geometries", "prison_clean.brep")
ASSETS_DIR = os.path.join(BASE_DIR, "assets")
os.makedirs(ASSETS_DIR, exist_ok=True)

GRID_SIZE = 2

# Image saving - the orchestrator sets this to True
SAVE_IMAGES = True

def save_fig(fig, filename):
    # Save a plotly figure to the assets directory as PNG.
    if not SAVE_IMAGES or fig is None:
        return
    try:
        path = os.path.join(ASSETS_DIR, filename)
        fig.write_image(path, width=1600, height=1200, scale=2)
        print(f"Saved: {path}")
    except Exception as e:
        print(f"Could not save {filename}: {e}")

## 4. Utility functions

In [ ]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)

## 5. Load the floor plan and set up the analysis grid

In [ ]:
# Load the cleaned floor plan
floor_plan = Topology.ByBREPPath(BREP_PATH)

# Create a grid overlay
b_r = Wire.BoundingRectangle(floor_plan)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange = list(range(0, int(width) + GRID_SIZE, GRID_SIZE))
vRange = list(range(0, int(length) + GRID_SIZE, GRID_SIZE))
grid = Grid.EdgesByDistances(floor_plan, clip=True, uRange=uRange, vRange=vRange)

# Slice the floor plan to create a shell
shell = Topology.Slice(floor_plan, grid)
faces = Topology.Faces(shell)
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_" + str(i + 1))
    f = Topology.SetDictionary(f, d)
print(f"Shell created with {len(faces)} faces")

## 6. Derive the analysis graph

In [ ]:
# Derive the analysis graph from the shell
analysis_graph = Graph.ByTopology(shell)
g_verts = Graph.Vertices(analysis_graph)
print(f"Analysis graph: {len(g_verts)} vertices, {len(Graph.Edges(analysis_graph))} edges")

## 7. Closeness Centrality (Integration)

Closeness centrality measures how close a node is to all other nodes. In space syntax terms, this corresponds to **global integration** - how accessible a space is within the overall configuration.

In [ ]:
centrality_list = Graph.ClosenessCentrality(analysis_graph, colorScale="thermal")

Transfer results back to the shell faces

In [ ]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

## 8. Show closeness centrality heatmap

In [ ]:
fig = Topology.Show(faces,
              faceColorKey="cc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              showFigure=False, renderer=renderer)
save_fig(fig, "07_closeness_centrality.png")

## 9. Betweenness Centrality (Choice)

Betweenness centrality measures how often a node lies on the shortest paths between other nodes. In space syntax, this corresponds to **choice** - spaces that act as critical circulation routes.

In [ ]:
centrality_list = Graph.BetweennessCentrality(analysis_graph, normalize=True, colorScale="thermal")

Transfer results back to the shell faces

In [ ]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

## 10. Show betweenness centrality heatmap

In [ ]:
fig = Topology.Show(faces,
              faceColorKey="bc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              showFigure=False, renderer=renderer)
save_fig(fig, "08_betweenness_centrality.png")